In [0]:
df = spark.table("bankingdata.silver.transaction")
display(df.limit(3))

In [0]:
df.columns

### 1. Fraud Spike Detection (Window-Based)

Business Use:
- Detect unusual activity in last 5 minutes

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

# Ensure timestamp
df = df.withColumn("event_time", col("event_time").cast("timestamp"))

fraud_spike_df = df \
    .withWatermark("event_time", "10 minutes") \
    .groupBy(
        col("account_id"),
        window(col("event_time"), "5 minutes")
    ) \
    .agg(
        count("*").alias("txn_count"),
        sum("amount").alias("total_amount"),
        countDistinct("location").alias("location_count")
    ) \
    .withColumn(
        "risk_flag",
        when(
            (col("txn_count") > 10) |
            (col("total_amount") > 20000) |
            (col("location_count") > 2),
            "HIGH"
        ).otherwise("NORMAL")
    )

In [0]:
fraud_spike_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("bankingdata.gold.fraud_spike_insights")

### 2. High-Value Transaction Anomaly

Business Use:
- Detect abnormal transaction compared to customer average

In [0]:
# Customer running average (streaming-safe approximation)
customer_avg_df = df.groupBy("customer_id") \
    .agg(avg("amount").alias("customer_avg"))

# Join streaming data with average
high_value_df = df.join(customer_avg_df, "customer_id", "left") \
    .withColumn(
        "is_anomaly",
        when(
            (col("amount") > 5000) &
            (col("amount") > col("customer_avg") * 3),
            True
        ).otherwise(False)
    ) \
    .withColumn(
        "action",
        when(col("is_anomaly") == True, "SEND_OTP").otherwise("ALLOW")
    )

In [0]:
high_value_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("bankingdata.gold.high_value_insights")

### 3. Customer Real-Time Behavior (Last Activity Snapshot)

Business Use:
- Track latest behavior for personalization

In [0]:
from pyspark.sql.window import Window

window_spec = Window.partitionBy("customer_id").orderBy(col("event_time").desc())

customer_behavior_df = df \
    .withColumn("rank", row_number().over(window_spec)) \
    .filter(col("rank") <= 10) \
    .groupBy("customer_id") \
    .agg(
        count("*").alias("recent_txn_count"),
        first("merchant_category").alias("top_category"),
        first("location").alias("recent_location")
    )

In [0]:
customer_behavior_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("bankingdata.gold.customer_behavior_insights")

### Save Gold Tables

In [0]:
fraud_spike_df.writeStream \
    .format("cosmos.oltp") \
    .options(**cosmos_config) \
    .option("checkpointLocation", "/mnt/checkpoints/fraud_spike") \
    .outputMode("update") \
    .start()

In [0]:
high_value_df.writeStream \
    .format("cosmos.oltp") \
    .options(**cosmos_config) \
    .option("spark.cosmos.container", "high_value_alerts") \
    .option("checkpointLocation", "/mnt/checkpoints/high_value") \
    .outputMode("append") \
    .start()

In [0]:
customer_behavior_df.writeStream \
    .format("cosmos.oltp") \
    .options(**cosmos_config) \
    .option("spark.cosmos.container", "customer_behavior") \
    .option("checkpointLocation", "/mnt/checkpoints/customer_behavior") \
    .outputMode("complete") \
    .start()